# Figure P05: phage ORFs of 10–90 aa against the GMSC smORFs

**Status:** draft, Gadi (agent-handoffs#27).

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figP05_auc_by_bin.tsv.gz` | Gadi `inphared_entropy/short_orfs_10_90/analysis{_code11,}/auc_by_bin.tsv`, `analysis_code11/gmsc_auc_by_bin.tsv` | agent-handoffs `inphared_gadi/short_orfs/52_short_orf_analysis.py` |
| `data/figP05_length_curve.tsv.gz` | `analysis_code11/length_curve.tsv.gz`, `gmsc_length_curve.tsv.gz` | `52_…`, `54_short_orf_figures.py` |
| `data/figP05_three_di_hist.tsv.gz`, `data/figP05_class_by_bin.tsv.gz` | `analysis_code11/` | `52_…` |
| `data/figP05_cds_length_bins.tsv.gz`, `data/figP05_matched_by_code.tsv.gz` | `analysis/` | `53_cds_lengths.py`, `56_matched_by_code.pbs` |
| `data/figP04_phage_length_3di.tsv.gz` (≥ 90 aa, panel b) | the P04 package | `prep/phage_p04_prepare_data.py` |

- **Reduction:** `manuscript/prep/phage_p05_prepare_data.py` copies the values; nothing is re-estimated.
- **Run:** agent-handoffs `inphared_gadi/short_orfs/` at 90f4fc6, PBS 181639266[0–9].
  - All 36,025 Inphared genomes were encoded again with `--min-aa 10 --max-aa 90`.
  - genome_entropy 0.4.0 (main 3597911, with the #103 fix), `gbouras13/modernprost-50M` (rev 2d68d506), encoding size 10,000, V100. Everything else is as in P04's run.
  - 102,133,250 ORFs, 0 failures.
- **Genomes:** 104 genomes have Pyrodigal-gv calls under genetic code 4 or 15. ORFs are called with table 11, so a reassigned stop splits one CDS into short ORFs that each match it.
  - These fragments are 95% of matched 10–29 aa ORFs, so panels a–c use the other **35,921 genomes** (101,414,229 ORFs).
  - The all-genome AUCs are the `genomes == "all"` rows. The ≥ 90 aa curve from P04 includes all 36,025 genomes.
- **AUC:** the Mann–Whitney statistic with average ranks for ties, matched (GMSC: positive) as the positive class, not folded about 0.5.
  - The GMSC rows were recomputed from `smorf_entropy.tsv.gz` with the same code and reproduce the smORF Table 2 values exactly.
- **3Di histograms:** 0.02-bit bins over 0–4.4 bits, normalised within each class and length bin.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")



In [ ]:
ac = pd.read_csv(DATA / "figP05_auc_by_bin.tsv.gz", sep="\t")
lc = pd.read_csv(DATA / "figP05_length_curve.tsv.gz", sep="\t")
hi = pd.read_csv(DATA / "figP05_three_di_hist.tsv.gz", sep="\t")
p04 = pd.read_csv(DATA / "figP04_phage_length_3di.tsv.gz", sep="\t")
COL = {"phage": "#CC79A7", "gmsc": "#E69F00"}
LAB = {("phage", "matched"): "phage, matched", ("phage", "unmatched"): "phage, unmatched",
       ("gmsc", "positive"): "GMSC positive", ("gmsc", "negative"): "GMSC negative"}
LS = {"matched": "-", "positive": "-", "unmatched": "--", "negative": "--"}

# Numbers quoted in the caption and the manuscript legend (README rule 5).
cb = pd.read_csv(DATA / "figP05_class_by_bin.tsv.gz", sep="\t")
assert cb.n.sum() == 101_414_229 and cb[cb.cls == "matched"].n.sum() == 654_673
cb10 = cb[cb["bin"] == "10-29"].set_index("cls").n
assert cb10["matched"] == 458 and round(cb10["unmatched"] / 1e6) == 71
g = ac[(ac.dataset == "gmsc") & (ac.bins == "gmsc_bins")]
assert round(g.n_pos.sum() / 1e6, 1) == 33.6 and round(g.n_neg.sum() / 1e6, 1) == 10.1


def auc(dset, genomes, col):
    t = ac[(ac.dataset == dset) & (ac.genomes == genomes) & (ac.bins == "gmsc_bins")]
    return t.set_index("bin")[col].round(3).to_dict()


ph3, php, gm3, gmp = (auc("phage", "code11", "three_di"), auc("phage", "code11", "protein"),
                      auc("gmsc", "gmsc", "three_di"), auc("gmsc", "gmsc", "protein"))
assert [ph3[b] for b in ("10-29", "30-49", "50-59", "80-89")] == [0.571, 0.575, 0.711, 0.848]
assert [php[b] for b in ("10-29", "30-49", "50-59", "80-89")] == [0.709, 0.698, 0.735, 0.840]
assert [gm3[b] for b in ("10-29", "30-49", "50-59", "90-99")] == [0.473, 0.608, 0.526, 0.626]
assert 0.455 <= min(gmp.values()) and max(gmp.values()) < 0.525
assert all(php[b] > ph3[b] for b in ("10-29", "30-49"))


def mean_70_90(dset, cls):
    t = lc[(lc.dataset == dset) & (lc.cls == cls) & lc.aa_length.between(70, 90)]
    return round((t.n * t["mean"]).sum() / t.n.sum(), 2)


assert [mean_70_90("phage_short", "matched"), mean_70_90("gmsc", "positive"),
        mean_70_90("phage_short", "unmatched"), mean_70_90("gmsc", "negative")] == [2.12, 1.87, 1.27, 1.62]
at90 = lc[(lc.dataset == "phage_short") & (lc.aa_length == 90)].set_index("cls")["mean"]
assert round(at90["matched"], 1) == 2.2 and round(at90["unmatched"], 2) == 1.28

fig = plt.figure(figsize=(7.2, 5.4))
gs = fig.add_gridspec(2, 4, height_ratios=[1.15, 1], hspace=0.6, wspace=0.45)

# (a) single-feature 3Di AUC by length bin, in GMSC's bins
ax = fig.add_subplot(gs[0, :2])
ORDER = ["10-29", "30-49", "50-59", "60-69", "70-79", "80-89", "90-99"]
for dset, genomes in (("phage", "code11"), ("gmsc", "gmsc")):
    t = ac[(ac.dataset == dset) & (ac.genomes == genomes) & (ac.bins == "gmsc_bins")]
    t = t.set_index("bin").reindex(ORDER).dropna(subset=["three_di"])
    x = [ORDER.index(b) for b in t.index]
    ax.plot(x, t.three_di, color=COL[dset], lw=1.2, marker="o", ms=3,
            label=f"{'phage' if dset == 'phage' else 'GMSC'}, 3Di entropy")
    ax.plot(x, t.protein, color=COL[dset], lw=0.8, ls=":", marker="s", ms=2,
            label=f"{'phage' if dset == 'phage' else 'GMSC'}, protein entropy")
ax.axhline(0.5, color=MUTED, lw=0.5, ls=(0, (1, 2)), zorder=0)
ax.set_xticks(range(len(ORDER)), [b.replace("-", "–") for b in ORDER], rotation=30, ha="right")
ax.set_xlabel("ORF length (aa)")
ax.set_ylabel("ROC AUC, single feature")
ax.set_ylim(0.4, 0.9)
ax.legend(frameon=False, fontsize=5.8, loc="upper left")
panel(ax, "a")

# (b) mean 3Di entropy by length, 10-90 aa joined to the >=90 aa run at 90 aa
ax = fig.add_subplot(gs[0, 2:])
for cls in ("matched", "unmatched"):
    s = lc[(lc.dataset == "phage_short") & (lc.cls == cls)]
    l = p04[(p04.dataset == "phage") & (p04.cls == cls) & (p04.aa_length > 90) & (p04.aa_length <= 300)]
    ax.plot(np.r_[s.aa_length, l.aa_length], np.r_[s["mean"], l["mean"]],
            color=COL["phage"], ls=LS[cls], lw=1.0, label=LAB[("phage", cls)])
for cls in ("positive", "negative"):
    g = lc[(lc.dataset == "gmsc") & (lc.cls == cls)]
    ax.plot(g.aa_length, g["mean"], color=COL["gmsc"], ls=LS[cls], lw=1.0, label=LAB[("gmsc", cls)])
ax.axvline(90, color=MUTED, lw=0.5, ls=(0, (1, 2)))
ax.axhline(np.log2(3), color=MUTED, lw=0.5, ls=(0, (1, 2)), zorder=0)
ax.set_xlim(10, 300)
ax.set_ylim(0, 3.2)
ax.set_xlabel("ORF length (aa)")
ax.set_ylabel("mean 3Di entropy (bits)")
ax.legend(frameon=False, fontsize=5.8, loc="upper left")
panel(ax, "b")

# (c) 3Di entropy distributions per length bin
W = 0.02
for j, b in enumerate(["10-29", "30-49", "50-69", "70-90"]):
    ax = fig.add_subplot(gs[1, j])
    for (dset, cls), lab in LAB.items():
        t = hi[(hi.dataset == dset) & (hi.cls == cls) & (hi["bin"] == b)]
        t = t.sort_values("three_di_left")
        edges = np.r_[t.three_di_left, t.three_di_left.iloc[-1] + W]
        assert np.allclose(np.diff(edges), W) and edges[0] == 0
        # stairs draws every bin from its left edge, including the zero-entropy bin [0, 0.02)
        ax.stairs(t.n / t.n.sum() / W, edges,
                  color=COL[dset], ls=LS[cls], lw=0.8, label=lab)
    ax.axvline(np.log2(3), color=MUTED, lw=0.5, ls=(0, (1, 2)))
    ax.set_xlim(0, 4.4)
    ax.set_title(f"{b.replace('-', '–')} aa", fontsize=7)
    ax.set_xlabel("3Di entropy (bits)")
    if j == 0:
        ax.set_ylabel("density")
        panel(ax, "c")
    if j == 3:
        ax.legend(frameon=False, fontsize=5.2, loc="upper right")
save(fig, "figP05_phage_short_orfs")


## Caption (draft)

**Figure P05. Below 50 aa, 3Di entropy separates short phage ORFs as weakly as it separates the GMSC smORFs. From 50 aa it separates them almost as well as gene-length phage ORFs, mostly because unannotated phage ORFs sit low.**

The phage data are ORFs of 10–90 aa from the 35,921 Inphared genomes whose genes are called with genetic code 11 (101.4 M ORFs, 654,673 matched to an annotated CDS). The GMSC data are 33.6 M positive and 10.1 M negative smORFs.

**(a)** Single-feature ROC AUC within length bins, for 3Di entropy (solid) and protein entropy (dotted).
- **Phage 3Di:** 0.571 at 10–29 aa, 0.575 at 30–49 aa, then 0.711 at 50–59 aa rising to 0.848 at 80–89 aa.
- **GMSC 3Di:** 0.473, 0.608, then 0.526 at 50–59 aa rising to 0.626 at 90–99 aa.
- **Phage protein entropy separates as well as 3Di entropy, and better below 50 aa** (0.709, 0.698, then 0.735 rising to 0.840). In GMSC protein entropy is near or below chance (0.46–0.52). Unlike GMSC's conserved negatives, the phage negatives differ from genes in composition as well as in predicted structure.
- **The 10–29 aa phage bin** holds only 458 annotated ORFs against 71 M unannotated ones.

**(b)** Mean 3Di entropy by ORF length. The phage 10–90 aa run (this figure) is joined at 90 aa (dotted line) to the ≥ 90 aa run of Figure P04; solid lines are matched or positive, dashed unmatched or negative. The two classes diverge from about 40 aa: annotated phage ORFs rise to 2.2 bits at 90 aa, while unannotated ORFs level off near 1.27 bits.

**(c)** 3Di entropy distributions in four length bins, each normalised within its class.

**Most of the difference from GMSC lies in the negatives.**
- **At 70–90 aa,** phage matched ORFs average 2.12 bits against 1.87 for GMSC positives. Phage unmatched ORFs average 1.27 bits against 1.62 for GMSC negatives.
- **The two negative sets differ by construction.** The phage negatives are every unannotated ORF, mostly antisense and frameshift shadows. The GMSC negatives are conserved, in-frame sequences selected to look coding.
